# LLM-Based Rubric Generation Workflow

使用 LLM 生成每个问题的分级评分标准，增强评估数据集。

**工作流程:**
1. 加载问题元数据和样本答案
2. 使用 LLM 基于分级样本答案生成评分标准
3. 保存和验证生成的评分标准

In [92]:
# Section 1: 导入必要的库和初始化 LLM 客户端

import pandas as pd
import json
import os
from pathlib import Path
from openai import OpenAI
from typing import Dict, List, Tuple, Union
import random
from datetime import datetime
from tqdm import tqdm

# Initialize OpenAI client
API_KEY = "REDACTED_OPENAI_KEY" # 用户提供的 API Key
client = OpenAI(api_key=API_KEY)

# Configuration - Updated for pt_asag dataset
DATA_DIR = "beetle"
MODEL = "gpt-4o-mini"
N_SAMPLES_PER_LEVEL = 10 
RANDOM_SEED = 42

print("✓ Libraries imported and OpenAI client initialized")
print(f"✓ Using dataset: {DATA_DIR}")


✓ Libraries imported and OpenAI client initialized
✓ Using dataset: beetle


## Section 2: 定义函数以检索问题元数据和样本答案

In [93]:
import glob

def load_data_from_csv(base_dir):
    """
    从 CSV 文件加载数据
    - train.csv: 训练集
    - test*.csv: 测试集（支持多个）
    """
    all_data = {}
    
    # 加载训练集
    train_path = os.path.join(base_dir, 'train.csv')
    if os.path.exists(train_path):
        all_data['training'] = pd.read_csv(train_path)
        print(f"训练集: {len(all_data['training'])} 条记录 ({train_path})")
    else:
        print(f"⚠️ 未找到训练集文件: {train_path}")
    
    # 加载所有测试集 (test*.csv)
    test_files = glob.glob(os.path.join(base_dir, 'test*.csv'))
    for test_path in sorted(test_files):
        test_name = os.path.basename(test_path).replace('.csv', '')
        all_data[test_name] = pd.read_csv(test_path)
        print(f"测试集 [{test_name}]: {len(all_data[test_name])} 条记录")
    
    return all_data

# 加载数据
print(f"正在从 {DATA_DIR} 加载 CSV 数据...")
all_data = load_data_from_csv(DATA_DIR)

# 训练数据列名和标签分布
if 'training' in all_data:
    print(f"\n训练数据列名: {all_data['training'].columns.tolist()}")
    if 'level' in all_data['training'].columns:
        print("标签分布:")
        print(all_data['training']['level'].value_counts())

# 加载问题元数据
print("\n正在加载问题元数据...")
meta_path = f'{DATA_DIR}/question_meta.json'
with open(meta_path, 'r', encoding='utf-8') as f:
    questions_meta = json.load(f)
print(f"成功加载了 {len(questions_meta)} 个问题的元数据")

# 显示数据集概览
print(f"\n{'='*50}")
print("数据集概览:")
print(f"{'='*50}")
for name, df in all_data.items():
    print(f"  {name}: {len(df)} 条记录")

def get_samples_by_question_and_label(question_id, label=None, n_samples=5):
    """
    Get sample student answers for a specific question and label
    """
    training_df = all_data['training']
    training_df["question_id"] = training_df["question_id"].astype(str)

    
    # Filter by question
    question_data = training_df[training_df['question_id'] == str(question_id)]
    
    if label:
        # Filter by label if specified
        question_data = question_data[question_data['level'] == label]
    
    # Sample n_samples entries
    samples = question_data.head(n_samples)
    
    # 获取问题文本 - 优先从 questions_meta 获取
    if str(question_id) in questions_meta:
        q_text = questions_meta[str(question_id)].get('question', '')
        sample_sol = questions_meta[str(question_id)].get('sample_solutions', 
                       questions_meta[str(question_id)].get('sample_solution', ''))
    else:
        q_text = question_data['question'].iloc[0] if not question_data.empty and 'question' in question_data.columns else ''
        sample_sol = question_data['sample_solution'].iloc[0] if not question_data.empty and 'sample_solution' in question_data.columns else ''
    
    result = {
        'question_id': question_id,
        'question_text': q_text,
        'sample_solution': sample_sol,
        'samples': []
    }
    
    for _, row in samples.iterrows():
        sample_entry = {
            'answer': row.get('answer_text', row.get('answer', '')),
            'level': row['level']
        }
        result['samples'].append(sample_entry)
    
    return result

# 测试函数
test_qid = list(questions_meta.keys())[0]
samples = get_samples_by_question_and_label(test_qid, n_samples=3)
print(f"\n=== 问题 {test_qid} 的样本数据 ===")
print(f"问题: {str(samples['question_text'])[:100]}...")
sample_sol = samples['sample_solution']
if isinstance(sample_sol, list):
    print(f"参考答案: {len(sample_sol)} 个")
else:
    print(f"参考答案: {str(sample_sol)[:100]}...")
print(f"\n学生答案样本:")
for i, sample in enumerate(samples['samples'], 1):
    print(f"{i}. 答案: {str(sample['answer'])[:100]}...")


正在从 beetle 加载 CSV 数据...
训练集: 3941 条记录 (beetle/train.csv)
测试集 [test_ua]: 439 条记录
测试集 [test_uq]: 819 条记录

训练数据列名: ['id', 'question_id', 'question', 'sample_solution', 'answer', 'level']
标签分布:
level
2    1665
0    1357
1     919
Name: count, dtype: int64

正在加载问题元数据...
成功加载了 56 个问题的元数据

数据集概览:
  training: 3941 条记录
  test_ua: 439 条记录
  test_uq: 819 条记录

=== 问题 FF.Q1 的样本数据 ===
问题: Explain why you got a voltage reading of 1.5 for terminal 1 and the positive terminal....
参考答案: Terminal 1 and the positive terminal are separated by the gap...

学生答案样本:
1. 答案: because terminal one and the positive terminal are connected...
2. 答案: because terminal one is connected to the positive battery terminal...
3. 答案: because terminal one and the positive battery terminal are on a closed path...


In [94]:
def get_question_metadata(qid):
    """
    Retrieve metadata for a specific question
    """
    qid_str = str(qid)
    if qid_str in questions_meta:
        return questions_meta[qid_str]
    else:
        return {}
def get_sample_answers(qid, n_samples_per_label=N_SAMPLES_PER_LEVEL):
    """
    Get sample answers grouped by label for a specific question
    Tries training data first, then test sets if no training data available
    """
    # First try training data
    if 'training' in all_data:
        training_df = all_data['training']
        # Handle both string and int question_id types
        question_data = training_df[training_df['question_id'] == qid]
        if question_data.empty and isinstance(qid, int):
            question_data = training_df[training_df['question_id'] == str(qid)]
        elif question_data.empty and isinstance(qid, str):
            try:
                question_data = training_df[training_df['question_id'] == int(qid)]
            except ValueError:
                pass
    else:
        question_data = pd.DataFrame()
    
    # If no training data, try test sets
    if question_data.empty:
        for key in all_data:
            if key.startswith('test'):
                print(f"  No training data for {qid}, trying {key}...")
                test_df = all_data[key]
                question_data = test_df[test_df['question_id'] == qid]
                if question_data.empty and isinstance(qid, int):
                    question_data = test_df[test_df['question_id'] == str(qid)]
                elif question_data.empty and isinstance(qid, str):
                    try:
                        question_data = test_df[test_df['question_id'] == int(qid)]
                    except ValueError:
                        pass
                if not question_data.empty:
                    break
    
    if question_data.empty:
        return {}
    
    # Get available labels from question metadata first
    question_info = get_question_metadata(qid)
    available_labels = question_info.get('level', [])
    
    # If no labels in metadata, get from data
    if not available_labels:
        available_labels = question_data['level'].unique()
        available_labels = [l for l in available_labels if pd.notna(l) and l != '']
    
    samples_by_label = {}
    
    for label in available_labels:
        label_data = question_data[question_data['level'] == label]
        # Get student answer column (handle different column names)
        sampled_answers = label_data["answer"].head(n_samples_per_label).tolist()
        
        if sampled_answers:
            samples_by_label[label] = sampled_answers
    
    return samples_by_label

print("✓ Helper functions defined (with CSV support and fallback to test sets)")


✓ Helper functions defined (with CSV support and fallback to test sets)


## Section 3: 创建 LLM 提示以生成分级评分标准

In [95]:
def label_sort_key(label: Union[str, int]) -> tuple:
    """Sort labels numerically when possible, otherwise lexicographically."""
    try:
        return (0, int(label))
    except (TypeError, ValueError):
        try:
            return (0, float(label))
        except (TypeError, ValueError):
            return (1, str(label))


def format_samples_for_prompt(samples_by_label: Dict[str, List[str]], label_order: List[Union[str, int]] = None) -> str:
    """Format sample answers for LLM prompt"""
    formatted = ""
    if label_order:
        ordered_labels = [label for label in label_order if label in samples_by_label] + \
                        [label for label in samples_by_label.keys() if label not in label_order]
    else:
        ordered_labels = list(samples_by_label.keys())
    
    for label in ordered_labels:
        answers = samples_by_label[label]
        formatted += f"\n**{label} Examples:**\n"
        for i, answer in enumerate(answers, 1):
            formatted += f"  {i}. {answer}\n"
    return formatted


def construct_system_prompt(available_labels: List[Union[str, int]] = None) -> str:
    """Construct system prompt for rubric generation"""
    label_texts = [str(l) for l in available_labels] if available_labels else []
    label_line = f"The feedback labels for this question are: {label_texts}\n" if label_texts else ""
    return f"""You are an expert educator specializing in assessment and rubric development. 
Your task is to analyze student answers and generate clear, criterion-based rubrics that distinguish between different feedback categories.

{label_line}You MUST generate a separate, specific rubric description for EACH feedback category that appears in the provided examples. Each rubric should be:
1. Specific and criterion-based - focus on observable behaviors and knowledge demonstration
2. Clearly differentiated - each category should be distinct from others
3. Clear and concise - use language that is easy to understand for educators
4. Evidence-based - grounded in the example responses provided
5. Educational - focus on what students demonstrate they know and can do
6. The rubric should be written in language used in the dataset. E.g. is the answers are in German, write the rubric in German.
CRITICAL: Generate individual rubric descriptions for each label category, not a general rubric."""


def construct_user_prompt(question_text: str, reference_answer: Union[str, List[str]], samples_by_label: Dict[str, List[str]], label_order: List[Union[str, int]] = None) -> str:
    """Construct user prompt for rubric generation"""
    formatted_samples = format_samples_for_prompt(samples_by_label, label_order=label_order)
    
    # Format reference answer(s)
    if isinstance(reference_answer, list):
        formatted_ref = "\n".join([f"  {i+1}. {ans}" for i, ans in enumerate(reference_answer)])
        ref_header = "**Sample Solutions / Reference Answers:**"
    else:
        formatted_ref = reference_answer
        ref_header = "**Reference Answer:**"
    
    # Get the actual labels present in the data
    present_labels = sorted(samples_by_label.keys(), key=label_sort_key)
    present_label_texts = [str(l) for l in present_labels]
    rubric_examples = "\n".join([
        f"        \"{label}\": \"Specific description for what characterizes label {label} based on the examples\""
        for label in present_label_texts
    ])
    
    prompt = f"""Based on the following question, reference answer(s), and student response examples with different feedback labels, 
generate a SEPARATE and SPECIFIC rubric description for EACH feedback category that describes what characterizes responses in that category.

**Question:**
{question_text}

{ref_header}
{formatted_ref}

**Student Response Examples by Feedback Category:**
{formatted_samples}

**CRITICAL REQUIREMENTS:**
You MUST generate a rubric description for each of these specific categories that appear in the data: {present_label_texts}

**Task:**
For EACH feedback category present in the examples above, write a detailed rubric description that explains:
1. What knowledge or understanding students demonstrate in this category
2. What specific characteristics distinguish responses in this category
3. What errors, gaps, or strengths are typical for this level
4. Clear criteria that help educators consistently categorize similar responses

Each rubric description should be:
- 2-4 sentences long
- Specific to that feedback level
- Based on the actual examples provided
- Actionable for educators

Return your response as a valid JSON object with the following format:
{{
    \"rubrics\": {{
{rubric_examples}
    }},
    \"reasoning\": \"Brief explanation of how you differentiated between the categories based on the examples\"
}}

IMPORTANT: 
- Only include rubrics for the feedback categories that have examples in the data: {present_label_texts}
- Use the labels exactly as provided above. If a label is numeric, return it as a numeric string (e.g., \"0\", \"1\").
- Each rubric must be specific to its category, not generic
- Base your rubrics on the actual student response patterns you observe in the examples
- Only return the JSON object, no additional text."""
    
    return prompt


print("✓ Prompt construction functions defined (with per-level specification and multi-solution support)")



✓ Prompt construction functions defined (with per-level specification and multi-solution support)


## Section 4: 实现核心函数：按问题 ID 生成评分标准

In [96]:
def generate_rubrics_for_question(qid: str, model: str = MODEL, verbose: bool = True) -> Dict:
    """
    Generate LLM-based rubrics for a question based on feedback labels
    
    Args:
        qid: question_id
        model: OpenAI model to use
        verbose: whether to print progress
    
    Returns:
        Dict with generated rubrics, reasoning, and metadata
    """
    try:
        # Get question metadata
        question_info = get_question_metadata(qid)
        question_text = question_info['question']
        
        # Handle multiple sample solutions if present
        reference_answer = question_info.get('sample_solutions', 
                                         question_info.get('reference_answer', 
                                                         question_info.get('sample_solution', '')))
        
        # Get sample answers by feedback label
        samples_by_label = get_sample_answers(qid)
        
        # Check if we have enough samples
        if not samples_by_label:
            raise ValueError(f"No valid samples found for question {qid}")
        label_order = sorted(samples_by_label.keys(), key=label_sort_key)

        # Construct prompts
        system_prompt = construct_system_prompt(label_order)
        user_prompt = construct_user_prompt(question_text, reference_answer, samples_by_label, label_order=label_order)
        
        # Call OpenAI API
        if verbose:
            print(f"  Calling LLM for {qid}... (Feedback labels: {list(samples_by_label.keys())})")
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_completion_tokens=1500
        )
        
        # Parse response
        response_text = response.choices[0].message.content
        
        # # 打印原始输出用于调试
        # if verbose:
        #     print(f"  Raw LLM response:")
        #     print(f"  {'-'*60}")
        #     print(response_text)
        #     print(f"  {'-'*60}")
        
        result = json.loads(response_text)
        
        # Prepare final result
        return {
            "question_id": qid,
            "question": question_text,
            "response": response_text,
            "reference_answer": reference_answer,
            "rubrics": result.get("rubrics", {}),
            "reasoning": result.get("reasoning", ""),
            "timestamp": datetime.now().isoformat(),
            "status": "success"
        }
    
    except json.JSONDecodeError as e:
        if verbose:
            print(f"  ✗ JSON parse error: {e}")
            if 'response_text' in locals():
                print(f"  Raw response that failed to parse:")
                print(f"  {'-'*60}")
                print(repr(response_text))
                print(f"  {'-'*60}")
        return {
            "question_id": qid,
            "response": response_text if 'response_text' in locals() else "",
            "status": "failed",
            "error": f"JSON parse error: {str(e)}",
            "timestamp": datetime.now().isoformat()
        }
    
    except Exception as e:
        if verbose:
            print(f"  ✗ Error: {e}")
            if 'response_text' in locals():
                print(f"  Raw response:")
                print(f"  {'-'*60}")
                print(repr(response_text))
                print(f"  {'-'*60}")
        return {
            "question_id": qid,
            "response": response_text if 'response_text' in locals() else "",
            "status": "failed",
            "error": str(e),
            "timestamp": datetime.now().isoformat()
        }


print("✓ Core generation function defined (with multi-solution handle and debug output)")



✓ Core generation function defined (with multi-solution handle and debug output)


### 测试单个问题的评分标准生成

In [97]:
# Test with single question from current dataset
test_qid = list(questions_meta.keys())[0]  # Get first available question

print(f"\n{'='*80}")
print(f"Testing rubric generation for: {test_qid}")
print(f"{'='*80}")

# Show question details first
question_info = get_question_metadata(test_qid)
samples = get_sample_answers(test_qid)

print(f"Question: {question_info['question']}")

# Handle reference answer display (can be list or string)
reference_answer = question_info.get('sample_solutions', 
                                   question_info.get('reference_answer', 
                                                   question_info.get('sample_solution', '')))

if isinstance(reference_answer, list):
    print(f"Reference Answers ({len(reference_answer)}):")
    for i, ans in enumerate(reference_answer[:3], 1): # Show first 3
        print(f"  {i}. {ans[:150]}...")
    if len(reference_answer) > 3:
        print(f"  ... and {len(reference_answer)-3} more")
else:
    print(f"Reference Answer: {reference_answer[:200]}...")

print(f"Available feedback labels: {list(samples.keys())}")
print(f"Sample counts: {[(label, len(answers)) for label, answers in samples.items()]}")

# Show some sample responses for each label
label_order = sorted(samples.keys(), key=label_sort_key)
ordered_labels = label_order

print(f"\nSample responses by label:")
for label in ordered_labels:
    answers = samples[label]
    print(f"\n{label} examples:")
    for i, answer in enumerate(answers[:N_SAMPLES_PER_LEVEL], 1):
        print(f"  {i}. {answer[:150]}...")

# Generate rubrics
result = generate_rubrics_for_question(test_qid)

if result['status'] == 'success':
    print(f"\n✓ Successfully generated rubrics!")
    print(f"\nGenerated Rubrics:")
    
    # Order labels for display
    available_rubrics = result['rubrics']
    label_order = sorted(available_rubrics.keys(), key=label_sort_key)
    ordered_labels = label_order
    
    for label in ordered_labels:
        if label in available_rubrics:
            print(f"\n  {label}:")
            print(f"  {available_rubrics[label]}")
            
    print(f"\nReasoning: {result['reasoning']}")
else:
    print(f"\n✗ Failed to generate rubrics")
    print(f"Error: {result.get('error', 'Unknown error')}")
    if result.get('response'):
        print(f"Response Text: {result.get('response', '')}")




Testing rubric generation for: FF.Q1
Question: Explain why you got a voltage reading of 1.5 for terminal 1 and the positive terminal.
Reference Answer: Terminal 1 and the positive terminal are separated by the gap...
Available feedback labels: [0, 2, 1]
Sample counts: [(0, 10), (2, 10), (1, 10)]

Sample responses by label:

0 examples:
  1. because terminal one and the positive terminal are connected...
  2. because terminal one is connected to the positive battery terminal...
  3. because terminal one and the positive battery terminal are on a closed path...
  4. because there was no separation in the positive battery terminal and terminal 1....
  5. because there was no gap in the positive battery terminal and terminal 1....
  6. because it was connected to the positive terminal...
  7. terminal 2 was connected to the negative terminal...
  8. its connected...
  9. because the terminals are in the same state...
  10. Terminal 1 is connected to the positive terminal....

1 examples:


## Section 5: 批量处理和结果保存

In [98]:
def generate_rubrics_batch(qids: List[str] = None, output_file: str = "safe_en/generated_rubrics.json", 
                           skip_existing: bool = True, verbose: bool = True) -> Dict:
    """
    Generate rubrics for multiple questions in batch
    
    Args:
        qids: List of question IDs (None = all)
        output_file: Path to save results
        skip_existing: Skip if rubric already exists
        verbose: Whether to print progress
    
    Returns:
        Dict with all generated rubrics
    """
    # Create output directory
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    # Load existing rubrics if they exist
    if os.path.exists(output_file) and skip_existing:
        with open(output_file, 'r', encoding='utf-8') as f:
            all_rubrics = json.load(f)
    else:
        all_rubrics = {}
    
    # Determine which qids to process
    if qids is None:
        qids = list(questions_meta.keys())
    
    # Filter out already processed qids if skip_existing
    if skip_existing:
        qids_to_process = [qid for qid in qids if qid not in all_rubrics or all_rubrics[qid].get('status') != 'success']
    else:
        qids_to_process = qids
    
    if verbose:
        print(f"\nGenerating rubrics for {len(qids_to_process)} questions...")
        print(f"(Skipping {len(qids) - len(qids_to_process)} already processed questions)")
        print(f"Questions to process: {qids_to_process}")
    
    # Process each question
    for i, qid in enumerate(tqdm(qids_to_process, desc="Generating rubrics"), 1):
        result = generate_rubrics_for_question(qid, verbose=False)
        all_rubrics[qid] = result
        # Save after every 3 questions (smaller batches for SAFE dataset)
        if (i % 3 == 0) or (i == len(qids_to_process)):
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(all_rubrics, f, indent=2, ensure_ascii=False)
            if verbose:
                successful = sum(1 for r in all_rubrics.values() if r.get('status') == 'success')
                print(f"  💾 Saved {len(all_rubrics)} rubrics ({successful} successful) to {output_file}")
    
    return all_rubrics


print("✓ Batch processing function defined")

✓ Batch processing function defined


## Section 6: 验证生成的评分标准质量

In [99]:
def validate_rubrics(rubrics_dict: Dict) -> pd.DataFrame:
    """
    Validate quality of generated rubrics
    
    Args:
        rubrics_dict: Dict of generated rubrics
    
    Returns:
        DataFrame with validation metrics
    """
    validation_results = []
    
    for qid, rubric_data in rubrics_dict.items():
        validation_row = {
            'question_id': qid,
            'status': rubric_data.get('status', 'unknown'),
            'has_rubrics': bool(rubric_data.get('rubrics')),
            'rubric_count': len(rubric_data.get('rubrics', {})),
            'has_reasoning': bool(rubric_data.get('reasoning')),
            'error': rubric_data.get('error', '')
        }
        
        # Additional checks
        if validation_row['has_rubrics']:
            rubrics = rubric_data.get('rubrics', {})
            # Check if all rubrics are non-empty strings
            validation_row['all_rubrics_non_empty'] = all(rubrics.values())
            # Check average rubric length
            avg_length = sum(len(r) if r else 0 for r in rubrics.values()) / len(rubrics) if rubrics else 0
            validation_row['avg_rubric_length'] = avg_length
        else:
            validation_row['all_rubrics_non_empty'] = False
            validation_row['avg_rubric_length'] = 0
        
        validation_results.append(validation_row)
    
    return pd.DataFrame(validation_results)


def print_validation_report(validation_df: pd.DataFrame):
    """Print validation report"""
    print(f"\n{'='*80}")
    print("VALIDATION REPORT")
    print(f"{'='*80}")
    
    total = len(validation_df)
    successful = len(validation_df[validation_df['status'] == 'success'])
    failed = len(validation_df[validation_df['status'] == 'failed'])
    
    print(f"\nTotal questions: {total}")
    print(f"  ✓ Successful: {successful} ({100*successful/total:.1f}%)")
    print(f"  ✗ Failed: {failed} ({100*failed/total:.1f}%)")
    
    if successful > 0:
        success_df = validation_df[validation_df['status'] == 'success']
        print(f"\nRubric Quality (successful questions):")
        print(f"  Average rubric length: {success_df['avg_rubric_length'].mean():.0f} characters")
        print(f"  All rubrics non-empty: {(success_df['all_rubrics_non_empty'].sum() / len(success_df) * 100):.1f}%")
        print(f"  Average levels per question: {success_df['rubric_count'].mean():.1f}")
    
    if failed > 0:
        print(f"\nFailure reasons:")
        failed_df = validation_df[validation_df['status'] == 'failed']
        error_counts = failed_df['error'].value_counts()
        for error, count in error_counts.head(5).items():
            print(f"  - {error[:80]}: {count}")


print("✓ Validation functions defined")

✓ Validation functions defined


## 完整工作流示例：生成和验证评分标准

以下示例展示如何使用上述函数为多个问题生成并验证评分标准。

In [100]:
# First, let's check what question IDs are actually available
print("Available question IDs in safe_en dataset:")
available_qids = list(questions_meta.keys())
print(f"Question IDs: {sorted(available_qids, key=label_sort_key)}")

# Example: Generate rubrics for ALL questions from SAFE dataset
print("\n" + "="*80)
print("EXAMPLE: Batch generation for SAFE dataset questions (Feedback-based)")
print("="*80)

# Use ALL available question IDs - no filtering!
print(f"\nAvailable questions: {sorted(available_qids)}")
print(f"Will generate rubrics for ALL {len(available_qids)} questions")

# Check training data availability for info only
print("\nTraining data availability (for information only):")
for qid in sorted(available_qids):
    training_df = all_data['training']
    question_data = training_df[training_df['question_id'] == str(qid)]
    if len(question_data) > 0:
        labels = question_data['level'].value_counts()
        print(f"Question {qid}: {len(question_data)} training samples - {dict(labels)}")
    else:
        print(f"Question {qid}: No training data (will try to generate rubrics anyway)")

# Generate rubrics for ALL questions regardless of training data availability
print(f"\nGenerating rubrics for ALL questions...")
all_rubrics = generate_rubrics_batch(
    qids=None,  # None means ALL questions
    output_file=f"{DATA_DIR}/generated_rubrics_feedback.json",
    skip_existing=False,
    verbose=True
)

# Validate results
validation_df = validate_rubrics(all_rubrics)
print_validation_report(validation_df)

# Display first successful rubric
print(f"\n{'='*80}")
print("SAMPLE GENERATED RUBRIC (FEEDBACK-BASED)")
print(f"{'='*80}")

successful_qids = [qid for qid, result in all_rubrics.items() if result.get('status') == 'success']
if successful_qids:
    first_qid = successful_qids[0]
    first_rubric = all_rubrics[first_qid]
    
    print(f"\nQuestion ID: {first_qid}")
    print(f"Question: {first_rubric['question']}")
    reference_answer = first_rubric.get('reference_answer', 'No reference answer available')
    print(f"Reference Answer: {reference_answer[:200]}...")
    print(f"\nAvailable Labels: {first_rubric.get('available_labels', [])}")
    print(f"Sample Counts: {first_rubric.get('samples_count', {})}")
    print(f"\nGenerated Rubrics:")
    
    rubrics = first_rubric.get('rubrics', {})
    # Order labels for display
    ordered_labels = sorted(rubrics.keys(), key=label_sort_key)
    
    for label in ordered_labels:
        if label in rubrics:
            print(f"\n  {label}:")
            print(f"  {rubrics[label]}")
        
    print(f"\nReasoning: {first_rubric.get('reasoning', 'No reasoning provided')}")
else:
    print("No successful rubrics generated. Check the error messages above.")

# Show summary of all results
print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")
print(f"Total questions processed: {len(all_rubrics)}")
successful_count = len([r for r in all_rubrics.values() if r.get('status') == 'success'])
failed_count = len([r for r in all_rubrics.values() if r.get('status') == 'failed'])
print(f"Successful: {successful_count}")
print(f"Failed: {failed_count}")

if failed_count > 0:
    print(f"\nFailed questions:")
    for qid, result in all_rubrics.items():
        if result.get('status') == 'failed':
            print(f"  {qid}: {result.get('error', 'Unknown error')}")


Available question IDs in safe_en dataset:
Question IDs: ['FF.Q1', 'FF.Q10', 'FF.Q11', 'FF.Q12', 'FF.Q13', 'FF.Q14', 'FF.Q15', 'FF.Q16', 'FF.Q17', 'FF.Q18', 'FF.Q19', 'FF.Q2', 'FF.Q20', 'FF.Q21', 'FF.Q22', 'FF.Q23', 'FF.Q3', 'FF.Q4', 'FF.Q5', 'FF.Q6', 'FF.Q7', 'FF.Q8', 'FF.Q9', 'PC.Q1', 'PC.Q10', 'PC.Q11', 'PC.Q12', 'PC.Q13', 'PC.Q14', 'PC.Q15', 'PC.Q16', 'PC.Q17', 'PC.Q18', 'PC.Q19', 'PC.Q2', 'PC.Q20', 'PC.Q21', 'PC.Q3', 'PC.Q4', 'PC.Q5', 'PC.Q6', 'PC.Q7', 'PC.Q8', 'PC.Q9', 'SC.Q1', 'SC.Q10', 'SC.Q11', 'SC.Q12', 'SC.Q2', 'SC.Q3', 'SC.Q4', 'SC.Q5', 'SC.Q6', 'SC.Q7', 'SC.Q8', 'SC.Q9']

EXAMPLE: Batch generation for SAFE dataset questions (Feedback-based)

Available questions: ['FF.Q1', 'FF.Q10', 'FF.Q11', 'FF.Q12', 'FF.Q13', 'FF.Q14', 'FF.Q15', 'FF.Q16', 'FF.Q17', 'FF.Q18', 'FF.Q19', 'FF.Q2', 'FF.Q20', 'FF.Q21', 'FF.Q22', 'FF.Q23', 'FF.Q3', 'FF.Q4', 'FF.Q5', 'FF.Q6', 'FF.Q7', 'FF.Q8', 'FF.Q9', 'PC.Q1', 'PC.Q10', 'PC.Q11', 'PC.Q12', 'PC.Q13', 'PC.Q14', 'PC.Q15', 'PC.Q16', 'PC.Q17', 'PC.Q

Generating rubrics:   5%|▌         | 3/56 [00:21<06:40,  7.55s/it]

  💾 Saved 3 rubrics (3 successful) to beetle/generated_rubrics_feedback.json


Generating rubrics:  11%|█         | 6/56 [00:43<05:55,  7.11s/it]

  💾 Saved 6 rubrics (6 successful) to beetle/generated_rubrics_feedback.json


...

  💾 Saved 54 rubrics (54 successful) to beetle/generated_rubrics_feedback.json
  No training data for SC.Q2, trying test_ua...
  No training data for SC.Q2, trying test_uq...


Generating rubrics:  98%|█████████▊| 55/56 [05:40<00:05,  5.96s/it]

  No training data for SC.Q6, trying test_ua...
  No training data for SC.Q6, trying test_uq...


Generating rubrics: 100%|██████████| 56/56 [05:46<00:00,  6.20s/it]

  💾 Saved 56 rubrics (56 successful) to beetle/generated_rubrics_feedback.json

VALIDATION REPORT

Total questions: 56
  ✓ Successful: 56 (100.0%)
  ✗ Failed: 0 (0.0%)

Rubric Quality (successful questions):
  Average rubric length: 418 characters
  All rubrics non-empty: 100.0%
  Average levels per question: 3.0

SAMPLE GENERATED RUBRIC (FEEDBACK-BASED)

Question ID: FF.Q1
Question: Explain why you got a voltage reading of 1.5 for terminal 1 and the positive terminal.
Reference Answer: Terminal 1 and the positive terminal are separated by the gap...

Available Labels: []
Sample Counts: {}

Generated Rubrics:

  0:
  Responses in this category demonstrate a fundamental misunderstanding of the relationship between terminal 1 and the positive terminal, often stating they are connected or in a closed path. These answers lack an understanding of the concept of separation and gaps, indicating confusion about electrical connections. Typical errors include misstatements of connection and conf